# Asistente experto en analitica de futbol profesional con Gemini, RAG y LangGraph

Proyecto final de IA Generativa: construccion de un agente experto capaz de responder preguntas sobre analitica de futbol profesional usando una base de conocimiento vectorial propia, Gemini como LLM, Gemini Embeddings, ChromaDB, LangGraph y memoria conversacional.

**Dominio elegido:** scouting y analitica avanzada de jugadores de futbol profesional en las cinco grandes ligas europeas durante la temporada 2024-2025.

## Requisitos del enunciado

Este notebook se construye para cumplir los puntos obligatorios del proyecto:

- Base de conocimiento vectorial en ChromaDB usando Gemini Embeddings.
- Minimo de 3 documentos o equivalente a unas 20 paginas de texto, usando CSV y documento propio.
- Pipeline RAG: ingesta, chunking/preprocesamiento, embeddings, almacenamiento, recuperacion y generacion.
- Agente con LangGraph que combine RAG, Gemini y memoria conversacional.
- System prompt personalizado y justificado.
- Celda de interaccion por texto dentro del notebook.
- Minimo de 5 preguntas de ejemplo con respuestas documentadas.

## 1. Instalacion de dependencias

Ejecuta esta celda solo si tu entorno no tiene instaladas las librerias. En local se recomienda usar `requirements.txt`; en Google Colab puedes descomentar la instalacion.

In [1]:
# Esta celda sirve para instalar las librerias si el entorno no las tiene.
# En local normalmente se instalan desde requirements.txt.
# En Google Colab conviene descomentar la linea siguiente y ejecutarla una vez.
!pip install -q langchain langchain-core langchain-community langchain-google-genai langchain-text-splitters langgraph chromadb python-dotenv pypdf pandas

## 2. Imports y configuracion del entorno

Las claves y configuraciones sensibles se cargan desde `.env`. No se debe hardcodear la API key en el notebook.

In [2]:
# Path permite trabajar con rutas de archivos de forma robusta en Windows, Linux o Colab.
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Cargamos el archivo .env para que Python pueda leer GOOGLE_API_KEY y otros parametros.
# override=True fuerza a usar el valor actual del .env aunque el kernel tuviera uno antiguo cargado.
load_dotenv(override=True)

# Leemos la clave de Gemini desde el .env.
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash").strip().strip('"')
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001").strip().strip('"')
CHROMA_DB_DIR = os.getenv("CHROMA_DB_DIR", "chroma_db").strip().strip('"')

# Si no hay API key, paramos el notebook con un error claro.
if not GOOGLE_API_KEY:
    raise ValueError("No se encontro GOOGLE_API_KEY. Revisa el archivo .env")

# Mostramos informacion de configuracion sin imprimir la API key completa.
print("API key cargada:", bool(GOOGLE_API_KEY))
print("Modelo Gemini:", GEMINI_MODEL)
print("Modelo embeddings:", GEMINI_EMBEDDING_MODEL)
print("Directorio Chroma:", CHROMA_DB_DIR)

c:\Users\User\miniconda3\envs\master_ds_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


API key cargada: True
Modelo Gemini: gemini-2.5-flash
Modelo embeddings: gemini-embedding-001
Directorio Chroma: chroma_db


## 3. Rutas del proyecto

Centralizamos las rutas para que el notebook sea mas mantenible y pueda ejecutarse desde la carpeta `notebooks` o desde la raiz del proyecto.

In [3]:
# Path.cwd() devuelve la carpeta desde la que se esta ejecutando el notebook.
NOTEBOOK_DIR = Path.cwd()

# Si ejecutamos el notebook desde la carpeta notebooks, la raiz del proyecto es el directorio padre.
# Si lo ejecutamos desde la raiz del proyecto, PROJECT_ROOT sera la carpeta actual.
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Carpeta donde estan los datos del proyecto.
DATA_DIR = PROJECT_ROOT / "data"

# Ruta del CSV con estadisticas de jugadores.
CSV_PATH = DATA_DIR / "players_data-2024_2025.csv"

# Lista de PDFs que formaran la base vectorial de conocimiento conceptual.
PDF_PATHS = sorted(DATA_DIR.glob("*.pdf"))

# Ruta donde ChromaDB guardara los vectores de forma persistente.
CHROMA_PATH = PROJECT_ROOT / CHROMA_DB_DIR

# Comprobamos que el CSV existe.
if not CSV_PATH.exists():
    raise FileNotFoundError(f"No se encontro el CSV requerido: {CSV_PATH}")

# Comprobamos que hay al menos 3 PDFs, como pide el enunciado.
if len(PDF_PATHS) < 3:
    raise FileNotFoundError("Se necesitan al menos 3 PDFs en la carpeta data para la base de conocimiento.")

# Imprimimos las rutas para verificar que el notebook esta leyendo los archivos correctos.
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV:", CSV_PATH)
print("PDFs encontrados:", len(PDF_PATHS))
for pdf_path in PDF_PATHS:
    print("-", pdf_path.name)
print("ChromaDB:", CHROMA_PATH)

PROJECT_ROOT: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve
CSV: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\data\players_data-2024_2025.csv
PDFs encontrados: 3
- 01_metricas_base_tiro_xg.pdf
- 02_metricas_pase_creacion_posesion.pdf
- 03_metricas_defensa_porteros_modelado_rag.pdf
ChromaDB: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\chroma_db


## 4. Inicializacion de Gemini

Creamos dos componentes: el modelo conversacional para generar respuestas y el modelo de embeddings para indexar y recuperar informacion desde ChromaDB.

In [4]:
# Creamos el modelo de lenguaje Gemini para generar respuestas.
llm = ChatGoogleGenerativeAI(
    # Modelo definido en .env, por ejemplo gemini-2.5-flash.
    model=GEMINI_MODEL,
    # API key cargada desde .env.
    google_api_key=GOOGLE_API_KEY,
    # Temperatura baja para respuestas mas consistentes y menos creativas.
    temperature=0.2,
)

# Creamos el modelo de embeddings que convertira texto en vectores numericos.
embeddings = GoogleGenerativeAIEmbeddings(
    # Modelo de embeddings definido en .env.
    model=GEMINI_EMBEDDING_MODEL,
    # API key cargada desde .env.
    google_api_key=GOOGLE_API_KEY,
)

# Confirmamos que ambos componentes se han creado.
print("LLM inicializado:", GEMINI_MODEL)
print("Embeddings inicializados:", GEMINI_EMBEDDING_MODEL)

LLM inicializado: gemini-2.5-flash
Embeddings inicializados: gemini-embedding-001


## 5. Prueba minima del LLM

Antes de construir el RAG, verificamos que Gemini responde correctamente. Esta prueba no usa todavia la base de conocimiento; solo valida conexion y configuracion.

In [5]:
# Enviamos una pregunta simple al LLM para comprobar que Gemini responde.
respuesta = llm.invoke("Explica en una frase que significa xG en futbol.")

# Imprimimos solo el contenido textual de la respuesta.
print(respuesta.content)

xG (Expected Goals) es una métrica que estima la probabilidad de que un disparo termine en gol, basándose en la calidad de la ocasión.


## 6. Carga inicial del dataset y de los PDFs

El CSV se usara con pandas para consultar datos exactos de jugadores. Los PDFs se usaran como documentos de conocimiento para ChromaDB y el RAG.

In [6]:
# Cargamos el CSV de jugadores en un DataFrame de pandas.
df = pd.read_csv(CSV_PATH)

# Mostramos dimensiones del dataset: filas y columnas.
print("Dataset:", df.shape)

# Mostramos el numero total de columnas.
print("Columnas:", len(df.columns))

# Mostramos los PDFs disponibles para la base vectorial.
print("Documentos PDF para RAG:")
for pdf_path in PDF_PATHS:
    print("-", pdf_path.name)

# Visualizamos algunas columnas importantes para comprobar que el CSV se cargo bien.
df[["Player", "Squad", "Comp", "Pos", "Age", "Min", "xG", "xAG", "PrgC", "PrgP"]].head()

Dataset: (2854, 267)
Columnas: 267
Documentos PDF para RAG:
- 01_metricas_base_tiro_xg.pdf
- 02_metricas_pase_creacion_posesion.pdf
- 03_metricas_defensa_porteros_modelado_rag.pdf


,Player,Squad,Comp,Pos,Age,Min,xG,xAG,PrgC,PrgP
0,Max Aarons,Bournemouth,eng Premier League,DF,24.0,86,0.0,0.0,1,8
1,Max Aarons,Valencia,es La Liga,"DF,MF",24.0,120,0.0,0.0,0,6
2,Rodrigo Abajas,Valencia,es La Liga,DF,21.0,65,0.1,0.0,3,2
3,James Abankwah,Udinese,it Serie A,"DF,MF",20.0,88,0.1,0.0,3,4
4,Keyliane Abdallah,Marseille,fr Ligue 1,FW,18.0,3,0.0,0.0,1,0


## 7. Revision rapida de cobertura de datos

Comprobamos que el dataset tiene suficiente volumen y variedad para justificar el proyecto: varias ligas, posiciones y miles de registros.

In [7]:
# Mostramos cuantas filas tiene el dataset.
print("Registros:", len(df))

# Contamos cuantos jugadores/registros hay por liga.
print("Ligas:")
display(df["Comp"].value_counts())

# Contamos las posiciones mas frecuentes del dataset.
print("Posiciones principales:")
display(df["Pos"].value_counts().head(15))

Registros: 2854
Ligas:


Comp
it Serie A            634
es La Liga            601
eng Premier League    574
fr Ligue 1            553
de Bundesliga         492
Name: count, dtype: int64

Posiciones principales:


Pos
DF       859
MF       589
FW       371
FW,MF    325
MF,FW    230
GK       212
DF,MF    110
MF,DF     81
DF,FW     53
FW,DF     24
Name: count, dtype: int64

## 8. Arquitectura MVP del asistente

Para mantener el proyecto simple y defendible, usaremos una arquitectura hibrida:

- **ChromaDB + Gemini Embeddings**: para recuperar conocimiento experto desde los 3 PDFs de metricas.
- **pandas + CSV completo**: para consultas numericas exactas sobre todos los jugadores del dataset.
- **Gemini LLM**: para redactar una respuesta clara usando el contexto recuperado y, cuando aplique, los resultados calculados desde el CSV.

Esta decision evita depender de similitud semantica para filtros exactos como `xG > 15`, `Min >= 1000` o `Top 10 por xAG`, que se resuelven mejor con pandas.

## 9. System prompt personalizado

El system prompt define el rol, tono, limites y reglas del asistente. Es una parte obligatoria del enunciado y se documentara tambien en el README.

In [8]:
# Definimos el system prompt principal del asistente.
# Este texto se enviara a Gemini para indicarle como debe comportarse.
SYSTEM_PROMPT = """
Eres un asistente experto en analitica de futbol profesional, especializado en scouting, rendimiento de jugadores y metricas avanzadas.

Tu objetivo es ayudar a analizar jugadores de futbol usando dos fuentes de informacion:
1. Contexto recuperado desde ChromaDB, especialmente el glosario de metricas y fichas de scouting indexadas.
2. Resultados calculados desde el CSV con pandas cuando la pregunta requiera rankings, filtros numericos o comparaciones exactas.

Reglas de comportamiento:
- Responde en español, con tono claro, profesional y didactico.
- No inventes datos de jugadores ni valores numericos.
- Si una respuesta requiere datos exactos y no se han proporcionado resultados del CSV, indica que necesitas consultar el dataset.
- Si el contexto recuperado no contiene informacion suficiente, dilo explicitamente.
- Explica las metricas cuando sean importantes para interpretar la respuesta.
- Cuando compares jugadores, ten en cuenta posicion, minutos jugados, liga y rol.
- Prioriza respuestas breves, utiles y basadas en evidencia.

Limitaciones:
- No predigas resultados futuros como si fueran certezas.
- No des recomendaciones de apuestas.
- No afirmes que un jugador es mejor que otro sin explicar con que metricas se justifica.
""".strip()

print(SYSTEM_PROMPT)

Eres un asistente experto en analitica de futbol profesional, especializado en scouting, rendimiento de jugadores y metricas avanzadas.

Tu objetivo es ayudar a analizar jugadores de futbol usando dos fuentes de informacion:
1. Contexto recuperado desde ChromaDB, especialmente el glosario de metricas y fichas de scouting indexadas.
2. Resultados calculados desde el CSV con pandas cuando la pregunta requiera rankings, filtros numericos o comparaciones exactas.

Reglas de comportamiento:
- Responde en español, con tono claro, profesional y didactico.
- No inventes datos de jugadores ni valores numericos.
- Si una respuesta requiere datos exactos y no se han proporcionado resultados del CSV, indica que necesitas consultar el dataset.
- Si el contexto recuperado no contiene informacion suficiente, dilo explicitamente.
- Explica las metricas cuando sean importantes para interpretar la respuesta.
- Cuando compares jugadores, ten en cuenta posicion, minutos jugados, liga y rol.
- Prioriza res

### 9.1 Justificacion del system prompt

Este prompt se ha diseñado para cumplir el enunciado porque:

- Define un rol experto concreto: analista profesional de futbol.
- Obliga a usar evidencia procedente de ChromaDB y del CSV.
- Reduce alucinaciones al prohibir inventar datos numericos.
- Explica como actuar si falta informacion.
- Mantiene un tono didactico, adecuado para un usuario que quiere entender metricas y decisiones de scouting.

## 10. Preparacion de documentos PDF para RAG

Para simplificar el proyecto, la base vectorial se construye solo con los 3 PDFs de metricas. Cada pagina del PDF se convierte en un documento de LangChain con metadatos de fuente y pagina.

In [9]:
# Document es la estructura de LangChain para representar texto + metadatos.
from langchain_core.documents import Document

# RecursiveCharacterTextSplitter divide textos largos en fragmentos mas pequenos.
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# Chroma es la base de datos vectorial que almacenara embeddings y documentos.
from langchain_community.vectorstores import Chroma

# PdfReader permite extraer texto de los PDFs.
from pypdf import PdfReader

### 10.1 Carga de paginas PDF como documentos

Extraemos el texto de cada pagina. Guardamos metadatos como nombre de archivo y numero de pagina para poder citar la fuente recuperada.

In [10]:
# Funcion auxiliar para corregir problemas tipicos de codificacion al extraer texto de algunos PDFs.
def repair_pdf_text(text):
    # Si aparecen caracteres como m??tricas o f??tbol, suele ser texto UTF-8 interpretado como Latin-1.
    if "?" in text or "?" in text:
        try:
            return text.encode("latin1").decode("utf-8")
        except UnicodeError:
            return text
    return text


# Lista donde guardaremos un Document por cada pagina de PDF.
pdf_page_documents = []

# Recorremos todos los PDFs encontrados en la carpeta data.
for pdf_path in PDF_PATHS:
    # Abrimos el PDF.
    reader = PdfReader(str(pdf_path))

    # Recorremos sus paginas.
    for page_number, page in enumerate(reader.pages, start=1):
        # Extraemos texto de la pagina. Si no hay texto, usamos cadena vacia.
        text = page.extract_text() or ""

        # Corregimos posibles problemas de codificacion y limpiamos espacios extremos.
        text = repair_pdf_text(text).strip()

        # Si una pagina no tiene texto, la saltamos.
        if not text:
            continue

        # Creamos un Document con el texto y sus metadatos.
        doc = Document(
            page_content=text,
            metadata={
                "source": pdf_path.name,
                "doc_type": "metric_pdf",
                "page": page_number,
            },
        )

        # A?adimos el documento a la lista.
        pdf_page_documents.append(doc)

# Mostramos resumen de documentos cargados.
print("Paginas PDF convertidas en documentos:", len(pdf_page_documents))
for doc in pdf_page_documents[:3]:
    print("-", doc.metadata, "|", doc.page_content[:120].replace("", " "))

Paginas PDF convertidas en documentos: 9
- {'source': '01_metricas_base_tiro_xg.pdf', 'doc_type': 'metric_pdf', 'page': 1} |  C o n t e x t o   d e   m é t r i c a s   f u t b o l í s t i c a s   p a r a   R A G   -   E v o l v e   A c a d e m y 
 P á g i n a   1 
   P D F   1   -   M é t r i c a s   b a s e ,   p r o d u c c i ó n   o f e n s i v a   y   t i r o 
- {'source': '01_metricas_base_tiro_xg.pdf', 'doc_type': 'metric_pdf', 'page': 2} |  C o n t e x t o   d e   m é t r i c a s   f u t b o l í s t i c a s   p a r a   R A G   -   E v o l v e   A c a d e m y 
 P á g i n a   2 
 M é t r i c a 
 D e f i n i c i ó n 
 U s o   p a r a   e l   a g e n t e 
 A d v e r t e n c i a 
 
- {'source': '01_metricas_base_tiro_xg.pdf', 'doc_type': 'metric_pdf', 'page': 3} |  C o n t e x t o   d e   m é t r i c a s   f u t b o l í s t i c a s   p a r a   R A G   -   E v o l v e   A c a d e m y 
 P á g i n a   3 
 N o t a   d e   m o d e l a d o :   p a r a   C h r o m a D B ,   c o n v i e r t 

## 11. Chunking de los PDFs

Dividimos las paginas en chunks mas pequenos. Esto mejora la recuperacion porque Chroma puede encontrar fragmentos concretos sobre xG, xA, PSxG, SCA, porteros, defensa, etc.

In [11]:
# Configuramos el divisor de texto.
text_splitter = RecursiveCharacterTextSplitter(
    # Tamano aproximado de cada chunk en caracteres.
    chunk_size=900,
    # Solapamiento entre chunks para no perder contexto entre cortes.
    chunk_overlap=120,
    # Separadores preferidos: parrafos, lineas, frases y palabras.
    separators=["", "", ". ", " ", ""],
)

# Dividimos las paginas de PDF en chunks.
pdf_chunks = text_splitter.split_documents(pdf_page_documents)

# Estos chunks son los documentos finales que se indexaran en ChromaDB.
rag_documents = pdf_chunks

# Mostramos el resultado del chunking.
print("Chunks PDF creados:", len(pdf_chunks))
print("Total documentos RAG:", len(rag_documents))
print(pdf_chunks[0].page_content[:800])
print("Metadatos:", pdf_chunks[0].metadata)

Chunks PDF creados: 37
Total documentos RAG: 37
Contexto de métricas futbolísticas para RAG - Evolve Academy
Página 1
 PDF 1 - Métricas base, producción ofensiva y tiro
Documento de contexto para que el agente entienda identidad del jugador, disponibilidad, goles, asistencias, xG y métricas
de finalización.
Dataset base: players_data-2024_2025.csv, 2854 jugadores y 267 columnas. Temporada 2024-2025.
Objetivo RAG: este documento está escrito para que el LLM recupere definiciones, interpretación y cautelas de cada
métrica antes de responder como analista profesional de fútbol.
Identidad y metadatos del jugador
Campos para identificar jugadores y contextualizar cualquier comparación.
Métrica
Definición
Uso para el agente
Advertencia
Rk
Ranking/fila original del jugador en la tabla fuente.
No usar como indicador deportivo; sirve
para trazab
Metadatos: {'source': '01_metricas_base_tiro_xg.pdf', 'doc_type': 'metric_pdf', 'page': 1}


## 12. Creacion de la base vectorial con ChromaDB

Creamos embeddings con Gemini para los chunks de los PDFs y los guardamos en ChromaDB. Esta base vectorial queda dedicada al conocimiento conceptual de metricas futbolisticas.

### 12.1 Prueba aislada de embeddings antes de indexar

Antes de enviar documentos a ChromaDB, probamos una sola consulta de embeddings. Si falla, el problema es de modelo, API key o cuota.

In [12]:
# Probamos el modelo de embeddings con un unico texto corto.
test_embedding = embeddings.embed_query("prueba de embeddings para metricas de futbol")

# Si funciona, deberia devolver una lista de numeros.
print("Dimension del embedding:", len(test_embedding))
print("Primeros valores:", test_embedding[:5])

Dimension del embedding: 3072
Primeros valores: [-0.0018784315, 0.017966228, 0.033197206, -0.08562882, 0.0038584026]


### 12.2 Indexacion en ChromaDB

Usamos una coleccion nueva solo para PDFs. Asi no mezclamos documentos antiguos de jugadores con la nueva arquitectura simplificada.

In [13]:
# Imports locales para que esta celda funcione aunque se ejecute de forma aislada.
import hashlib
import time

# Nombre de la coleccion dentro de ChromaDB.
COLLECTION_NAME = "asistente_metricas_futbol_pdf"

# Creamos una instancia de ChromaDB usando Gemini Embeddings.
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH),
)

# Funcion auxiliar para crear IDs estables a partir de fuente, pagina e indice.
def build_document_id(doc, index):
    raw_id = "|".join([
        str(doc.metadata.get("source", "")),
        str(doc.metadata.get("page", "")),
        str(index),
    ])
    return hashlib.md5(raw_id.encode("utf-8")).hexdigest()

# Lotes pequenos para evitar problemas de cuota.
BATCH_SIZE = 5
SLEEP_SECONDS = 8

# Creamos IDs para todos los chunks.
document_ids = [build_document_id(doc, i) for i, doc in enumerate(rag_documents)]

# Obtenemos IDs ya existentes para no duplicar documentos si reejecutamos la celda.
existing = vectorstore._collection.get(include=[])
existing_ids = set(existing.get("ids", []))

# Insertamos chunks por lotes.
for start in range(0, len(rag_documents), BATCH_SIZE):
    end = start + BATCH_SIZE
    batch_docs = rag_documents[start:end]
    batch_ids = document_ids[start:end]

    pending_pairs = [
        (doc, doc_id)
        for doc, doc_id in zip(batch_docs, batch_ids)
        if doc_id not in existing_ids
    ]

    if not pending_pairs:
        print(f"Lote {start + 1}-{min(end, len(rag_documents))} ya estaba indexado")
        continue

    pending_docs = [pair[0] for pair in pending_pairs]
    pending_ids = [pair[1] for pair in pending_pairs]

    try:
        vectorstore.add_documents(documents=pending_docs, ids=pending_ids)
        existing_ids.update(pending_ids)
        print(f"Indexados documentos {start + 1}-{min(end, len(rag_documents))} de {len(rag_documents)}")
    except Exception as error:
        print("Se detuvo la indexacion por un error, probablemente cuota de Gemini.")
        print("Documentos guardados hasta ahora:", vectorstore._collection.count())
        print("Error:", error)
        break

    if end < len(rag_documents):
        time.sleep(SLEEP_SECONDS)

print("Coleccion creada/cargada:", COLLECTION_NAME)
print("Directorio persistente:", CHROMA_PATH)
print("Documentos en ChromaDB:", vectorstore._collection.count())

C:\Users\User\AppData\Local\Temp\ipykernel_23112\2527730277.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Indexados documentos 1-5 de 37
Indexados documentos 6-10 de 37
Indexados documentos 11-15 de 37
Indexados documentos 16-20 de 37
Indexados documentos 21-25 de 37
Indexados documentos 26-30 de 37
Indexados documentos 31-35 de 37
Indexados documentos 36-37 de 37
Coleccion creada/cargada: asistente_metricas_futbol_pdf
Directorio persistente: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\chroma_db
Documentos en ChromaDB: 37


### 12.3 Recarga de la coleccion existente

En futuras ejecuciones, puedes saltar la indexacion y ejecutar solo esta celda para cargar la coleccion ya guardada.

In [14]:
# Nombre de la coleccion dedicada a los PDFs de metricas.
COLLECTION_NAME = "asistente_metricas_futbol_pdf"

# Recargamos la coleccion existente sin recalcular embeddings.
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH),
)

print("Coleccion cargada:", COLLECTION_NAME)
print("Documentos disponibles:", vectorstore._collection.count())

Coleccion cargada: asistente_metricas_futbol_pdf
Documentos disponibles: 37


## 13. Prueba del retriever

Antes de crear el agente, comprobamos que ChromaDB recupera fragmentos relevantes de los PDFs.

In [15]:
# Convertimos ChromaDB en un retriever de LangChain.
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# Funcion auxiliar para probar recuperacion semantica.
def probar_retriever(pregunta):
    docs = retriever.invoke(pregunta)
    print("Pregunta:", pregunta)
    print("Documentos recuperados:", len(docs))

    for i, doc in enumerate(docs, start=1):
        print("" + "=" * 80)
        print(f"Documento {i}")
        print("Fuente:", doc.metadata.get("source"))
        print("Pagina:", doc.metadata.get("page"))
        print("Tipo:", doc.metadata.get("doc_type"))
        print(doc.page_content[:900])

    return docs

In [16]:
docs_prueba_metricas = probar_retriever("Que significa xG y como se interpreta en scouting?")

Pregunta: Que significa xG y como se interpreta en scouting?
Documentos recuperados: 5
Documento 1
Fuente: 01_metricas_base_tiro_xg.pdf
Pagina: 1
Tipo: metric_pdf
; no sobreinterpretar.
xG
Goles esperados: suma de probabilidades de gol
de los tiros.
Evaluar calidad de ocasiones generadas
por el jugador.
Los modelos cambian por
proveedor; no mezclar fuentes.
Documento 2
Fuente: 01_metricas_base_tiro_xg.pdf
Pagina: 2
Tipo: metric_pdf
enen el mismo valor.
PrgR
Recepciones progresivas.
Capacidad de recibir en zonas
avanzadas.
Muy relevante en extremos y
delanteros.
G+A-PK
Goles + asistencias excluyendo penaltis marcados.
Producción directa ajustada por penaltis.
No elimina asistencias de penaltis
provocados.
xG+xAG
xG + xAG total.
Resumen de amenaza esperada total.
Incluye penaltis dentro de xG.
Métricas de tiro y finalización
Indicadores para analizar volumen, precisión, selección de tiro y sobre/subrendimiento.
Métrica
Definición
Uso para el agente
Advertencia
Sh
Tiros totales.
Volumen d

In [17]:
docs_prueba_porteros = probar_retriever("Que significa PSxG+/- y como se usa para evaluar porteros?")

Pregunta: Que significa PSxG+/- y como se usa para evaluar porteros?
Documentos recuperados: 5
Documento 1
Fuente: 03_metricas_defensa_porteros_modelado_rag.pdf
Pagina: 3
Tipo: metric_pdf
Contexto de métricas futbolísticas para RAG - Evolve Academy
Página 3
Métrica
Definición
Uso para el agente
Advertencia
CK
Goles recibidos desde córner.
Defensa de balón parado.
Responsabilidad compartida.
OG
Autogoles recibidos a favor del rival.
Contexto de GA.
No atribuir al GK.
PSxG
Post-Shot xG: calidad de tiros a puerta recibidos
tras conocer colocación del tiro.
Evaluar dificultad real de paradas.
Fuente/modelo específico.
PSxG/SoT
PSxG medio por tiro a puerta.
Dificultad media de los tiros recibidos.
Alto valor implica tiros difíciles.
PSxG+/-
PSxG menos goles recibidos.
Paradas sobre lo esperado: positivo suele
indicar buen rendimiento.
Requiere muestra alta.
/90
PSxG+/- por 90.
Rendimiento de parada normalizado.
Nombre de columna ambiguo;
renombrar en preprocesado.
Cmp
Pases largos completad

## 14. RAG basico con Gemini

Ahora conectamos el retriever con Gemini. Esta version todavia no usa LangGraph ni memoria; sirve para comprobar que el modelo responde usando contexto recuperado desde ChromaDB.

In [18]:
# Funcion para convertir documentos recuperados en texto legible para el prompt.
def format_retrieved_docs(docs):
    # Creamos una lista donde guardaremos cada documento formateado.
    formatted = []

    # Recorremos los documentos recuperados por ChromaDB.
    for i, doc in enumerate(docs, start=1):
        # Extraemos metadatos utiles.
        source = doc.metadata.get("source", "fuente desconocida")
        doc_type = doc.metadata.get("doc_type", "tipo desconocido")
        page = doc.metadata.get("page")

        # Creamos una cabecera para saber de donde viene cada fragmento.
        header = f"[Fragmento recuperado {i} | fuente: {source} | tipo: {doc_type}"
        if page:
            header += f" | pagina: {page}"
        header += "]"

        # Juntamos cabecera y contenido del documento.
        formatted.append(header + "" + doc.page_content)

    # Devolvemos todos los documentos separados por una linea visual.
    return "---".join(formatted)


# Funcion principal de RAG basico.
def responder_con_rag_basico(pregunta, k=5):
    # Recuperamos documentos relevantes desde ChromaDB.
    docs = retriever.invoke(pregunta)

    # Nos quedamos con los k primeros por si el retriever devuelve mas.
    docs = docs[:k]

    # Convertimos los documentos recuperados en texto para Gemini.
    contexto = format_retrieved_docs(docs)

    # Creamos el prompt final con instrucciones, pregunta y contexto.
    prompt = f"""
{SYSTEM_PROMPT}

Pregunta del usuario:
{pregunta}

Contexto recuperado desde ChromaDB:
{contexto}

Instrucciones para responder:
- Responde usando el contexto recuperado.
- Si el contexto no contiene la informacion necesaria, dilo claramente.
- Cita brevemente las fuentes usando el nombre del PDF y la pagina indicada en los metadatos.
""".strip()

    # Enviamos el prompt a Gemini.
    respuesta = llm.invoke(prompt)

    # Devolvemos respuesta y documentos para poder inspeccionar el RAG.
    return {
        "respuesta": respuesta.content,
        "documentos": docs,
        "prompt": prompt,
    }

### 14.1 Prueba conceptual del RAG

Esta prueba debe recuperar fragmentos del glosario y responder usando definiciones de metricas.

In [19]:
resultado_xg = responder_con_rag_basico("Que significa xG y como se interpreta en scouting?")
print(resultado_xg["respuesta"])

El xG, o Goles Esperados, es una métrica avanzada que representa la suma de las probabilidades de gol de los tiros realizados por un jugador o equipo. Cada tiro tiene una probabilidad de convertirse en gol, calculada en función de diversos factores como la distancia al arco, el ángulo, el tipo de asistencia, la parte del cuerpo utilizada, y la situación del juego (por ejemplo, si es un contraataque o un tiro a balón parado).

**Interpretación en scouting:**

En el scouting, el xG se utiliza principalmente para:

1.  **Evaluar la calidad de las ocasiones generadas:** Permite entender si un jugador está llegando a posiciones de remate de alta calidad, independientemente de si el tiro termina en gol o no. Un jugador con un alto xG está generando buenas oportunidades de gol (Fragmento recuperado 1 | fuente: 01_metricas_base_tiro_xg.pdf | pagina: 1).
2.  **Analizar el volumen y la calidad de los tiros:** Al comparar el xG con los goles reales (G-xG), se puede identificar si un jugador está 

### 14.2 Inspeccion de documentos usados

Esta celda permite demostrar que la respuesta no sale solo del LLM, sino de documentos recuperados por ChromaDB.

In [20]:
for i, doc in enumerate(resultado_xg["documentos"], start=1):
    print("=" * 80)
    print(f"Documento recuperado {i}")
    print("Metadatos:", doc.metadata)
    print(doc.page_content[:800])

Documento recuperado 1
Metadatos: {'page': 1, 'source': '01_metricas_base_tiro_xg.pdf', 'doc_type': 'metric_pdf'}
; no sobreinterpretar.
xG
Goles esperados: suma de probabilidades de gol
de los tiros.
Evaluar calidad de ocasiones generadas
por el jugador.
Los modelos cambian por
proveedor; no mezclar fuentes.
Documento recuperado 2
Metadatos: {'doc_type': 'metric_pdf', 'source': '01_metricas_base_tiro_xg.pdf', 'page': 2}
enen el mismo valor.
PrgR
Recepciones progresivas.
Capacidad de recibir en zonas
avanzadas.
Muy relevante en extremos y
delanteros.
G+A-PK
Goles + asistencias excluyendo penaltis marcados.
Producción directa ajustada por penaltis.
No elimina asistencias de penaltis
provocados.
xG+xAG
xG + xAG total.
Resumen de amenaza esperada total.
Incluye penaltis dentro de xG.
Métricas de tiro y finalización
Indicadores para analizar volumen, precisión, selección de tiro y sobre/subrendimiento.
Métrica
Definición
Uso para el agente
Advertencia
Sh
Tiros totales.
Volumen de finalizac

### 14.3 Prueba sobre metricas de porteros

Como ChromaDB ahora solo contiene PDFs conceptuales, esta prueba valida una pregunta sobre metricas de porteros. Los jugadores concretos se consultaran despues con pandas.

In [21]:
resultado_porteros = responder_con_rag_basico("Explica PSxG, PSxG/SoT y PSxG+/- para evaluar porteros")
print(resultado_porteros["respuesta"])

Para evaluar porteros con métricas avanzadas, PSxG, PSxG/SoT y PSxG+/- son indicadores clave que permiten ir más allá de las paradas básicas y entender la calidad de las intervenciones.

Aquí te explico cada una:

1.  **PSxG (Post-Shot xG)**
    *   **Definición:** Representa la calidad de los tiros a puerta recibidos *después* de conocer la colocación y potencia del disparo. A diferencia del xG pre-tiro, el PSxG tiene en cuenta cómo fue el remate final, lo que lo hace más preciso para evaluar la dificultad real de una parada.
    *   **Uso:** Permite evaluar la dificultad real de las paradas que un portero enfrenta. Un valor alto de PSxG en un tiro indica que era un disparo muy difícil de detener.
    *   **Advertencia:** Su cálculo depende de la fuente y el modelo específico utilizado para generarlo.
    *   *Fuente: 03_metricas_defensa_porteros_modelado_rag.pdf, página 3*

2.  **PSxG/SoT (PSxG medio por tiro a puerta)**
    *   **Definición:** Es el valor promedio de PSxG por cada t

## Resultado de esta fase

En este punto ya tenemos:

- 3 PDFs de metricas cargados como documentos de conocimiento.
- PDFs divididos en chunks semanticos.
- Embeddings creados con Gemini.
- Base vectorial persistente en ChromaDB.
- Retriever probado antes de conectar el agente.
- RAG basico con Gemini usando contexto recuperado desde los PDFs.

A partir de aqui, el CSV completo se usara con pandas para consultar jugadores, rankings y filtros numericos exactos. Esta separacion hace el proyecto mas simple y fiable: ChromaDB explica metricas; pandas consulta datos de jugadores.